<a href="https://colab.research.google.com/github/iamuhd/my_ML_Intership_at_FLYRANK-AI/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections in order — each one has a one-line hint. Simple words, honest numbers.

> We are working in the FlyRank refresh / content-opportunity lane. We need a contract that says exactly what a row means and which columns are safe to use before the prediction moment.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row is one pseudonymized content item (`content_id`). This is a page-level dataset where each row summarizes one article/page over the trailing 90-day window ending at export time. The dictionary explicitly says metrics are trailing-90-day totals, and the 30-day comparison columns are the most recent 30 days versus the prior 30 days.

In [2]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path.cwd() / 'data' / 'raw' / 'content_refresh_anonymized.csv',
    Path.cwd() / 'content_refresh_anonymized.csv',
    Path.cwd().parent / 'data' / 'raw' / 'content_refresh_anonymized.csv',
]

data_path = next((p for p in candidate_paths if p.exists()), candidate_paths[0])
df = pd.read_csv(data_path)

print('Loaded file:', data_path)
print('Rows:', len(df))
print('Unique content ids:', df['content_id'].nunique())
print('Unique clients:', df['client_id'].nunique())
print('One row per content item?', df['content_id'].is_unique)
print(df[['content_id', 'client_id', 'content_type', 'impressions_90d', 'ctr', 'avg_position', 'trend_direction']].head(5).to_string(index=False))


Loaded file: /content/content_refresh_anonymized.csv
Rows: 30000
Unique content ids: 30000
Unique clients: 32
One row per content item? True
          content_id         client_id    content_type  impressions_90d  ctr  avg_position trend_direction
content_304f48230142 client_f369cb89fc keyword article             3803 0.76          10.6            down
content_a1fb4e703a9e client_4e07408562 keyword article            15320 0.05          20.3            down
content_9aa793d4d895 client_7f2253d7e2 keyword article            12581 0.09          36.5            down
content_331d6c4de07b client_19581e27de keyword article            11751 0.49           6.2          stable
content_d99b7a2d90ca client_3fdba35f04 keyword article            19140 0.13          44.0            down


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

I will touch the page-level signal columns, the decline label, and a few grouping columns for validation.

- Feature: `search_volume`, `competition`, `competition_level`, `cpc`, `word_count`, `char_count`, `content_age_days`, `days_since_last_update`, `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions`, `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`
- Label / proxy: `trend_direction`, `trend_pct`, `is_declining_label`
- Context: `content_id`, `client_id`, `content_type`, `main_intent`
- Excluded: `provider_used`, `model_used` — these are generation metadata, not stable predictive content features, and they can leak workflow identity rather than page quality.

In [3]:
feature_cols = [
    'search_volume', 'competition', 'competition_level', 'cpc',
    'word_count', 'char_count', 'content_age_days', 'days_since_last_update',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct'
]

label_cols = ['trend_direction', 'trend_pct', 'is_declining_label']
context_cols = ['content_id', 'client_id', 'content_type', 'main_intent']
excluded_cols = ['provider_used', 'model_used']

bucket_map = {c: 'feature' for c in feature_cols}
bucket_map.update({c: 'label' for c in label_cols})
bucket_map.update({c: 'context' for c in context_cols})
bucket_map.update({c: 'excluded' for c in excluded_cols})

field_df = pd.DataFrame({
    'column': list(bucket_map.keys()),
    'bucket': list(bucket_map.values())
}).sort_values(['bucket', 'column'], ignore_index=True)

print(field_df.to_string(index=False))
print('\nExcluded reason: provider_used and model_used are generation metadata, not stable page-quality predictors.')


                column   bucket
             client_id  context
            content_id  context
          content_type  context
           main_intent  context
            model_used excluded
         provider_used excluded
       ai_sessions_90d  feature
        ai_traffic_pct  feature
          avg_position  feature
            char_count  feature
            clicks_90d  feature
       clicks_last_30d  feature
       clicks_prev_30d  feature
           competition  feature
     competition_level  feature
      content_age_days  feature
                   cpc  feature
                   ctr  feature
days_since_last_update  feature
 days_with_impressions  feature
    days_with_sessions  feature
  engaged_sessions_90d  feature
       engagement_rate  feature
       impressions_90d  feature
  impressions_last_30d  feature
  impressions_prev_30d  feature
         pageviews_90d  feature
     scroll_events_90d  feature
           scroll_rate  feature
         search_volume  feature
        

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

My three direct checks are: (1) grain check, (2) counts / client coverage, and (3) missingness + window sanity. The five-feature trap is the set of obvious leakage columns: `trend_direction`, `trend_pct`, `is_declining_label`, `impressions_last_30d`, and `sessions_last_30d`. A model must never learn from columns that are built from the same outcome logic or from the same label window.

In [6]:
content_counts = df.groupby('content_id').size().reset_index(name='n_rows')
print('Duplicate content ids:', int((content_counts['n_rows'] > 1).sum()))
print('One row per content item?', df['content_id'].is_unique)

# Query 2: Counts and client coverage
print('Rows in slice:', len(df))
print('Unique clients:', df['client_id'].nunique())
print('Label rate (trend_direction == down):', round((df['trend_direction'] == 'down').mean(), 4))

# Check for 'is_declining_label' before using it
if 'is_declining_label' in df.columns:
    print('Proxy rate (is_declining_label == 1):', round(df['is_declining_label'].mean(), 4))
else:
    print("Warning: Column 'is_declining_label' not found in DataFrame. Skipping 'Proxy rate' calculation.")

# Query 3: Missingness and window sanity
missing_by_type = df.groupby('content_type')[['search_volume', 'word_count']].apply(lambda x: x.isna().mean().round(4))
print('Missingness by content_type:')
print(missing_by_type.to_string())

print('avg_position == 0 count:', int((df['avg_position'] == 0).sum()))
print('avg_position == 0 share:', round((df['avg_position'] == 0).mean(), 4))
print('Window summary:')
print(df[['impressions_last_30d', 'impressions_prev_30d', 'sessions_last_30d', 'sessions_prev_30d', 'content_age_days']].describe().round(3).to_string())

# The five-feature trap
# Dynamically build five_feature_trap based on column existence
five_feature_trap = ['trend_direction', 'trend_pct', 'impressions_last_30d', 'sessions_last_30d']
if 'is_declining_label' in df.columns:
    five_feature_trap.append('is_declining_label')
else:
    print("Warning: Column 'is_declining_label' not found. Excluded from 'five_feature_trap'.")

print('Five-feature trap candidates:')
print(five_feature_trap)
print(df[five_feature_trap].head().to_string(index=False))

Duplicate content ids: 0
One row per content item? True
Rows in slice: 30000
Unique clients: 32
Label rate (trend_direction == down): 0.5421
Missingness by content_type:
                    search_volume  word_count
content_type                                 
comparison article         0.0000       0.000
feedly article             1.0000       0.000
keyword article            0.0137       0.283
avg_position == 0 count: 1205
avg_position == 0 share: 0.0402
Window summary:
       impressions_last_30d  impressions_prev_30d  sessions_last_30d  sessions_prev_30d  content_age_days
count             30000.000             30000.000          30000.000          30000.000         30000.000
mean               1429.059              1783.078             14.114             10.283           256.168
std                5643.852              6150.430             39.036             42.578           132.708
min                   0.000                 0.000              0.000              0.000           

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset is a static content slice, not a full causal panel. It cannot tell us whether a decline is caused by weak content quality, a shift in search demand, or a client-specific seasonal effect. The panel is also unbalanced: some content items have very long histories and some have sparse or missing metadata. A blind fill would silently encode `content_type` differences, and the trend label is built from a short comparison window, so it is directional evidence, not a final causal truth.

In [7]:
print('Rows with missing search_volume:', int(df['search_volume'].isna().sum()))
print('Rows with missing word_count:', int(df['word_count'].isna().sum()))
print('Rows with avg_position == 0:', int((df['avg_position'] == 0).sum()))
print('Rows with trend_pct missing because prev period was zero:', int(df['trend_pct'].isna().sum()))
print('content_age_days min / max:', df['content_age_days'].min(), df['content_age_days'].max())
print('content_type counts:')
print(df['content_type'].value_counts().to_string())


Rows with missing search_volume: 2468
Rows with missing word_count: 7699
Rows with avg_position == 0: 1205
Rows with trend_pct missing because prev period was zero: 3388
content_age_days min / max: 90 564
content_type counts:
content_type
keyword article       27207
feedly article         2096
comparison article      697


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom and shows the row definition, field buckets, and verification checks
- [x] The unit is explicit: one content item, one row, trailing 90-day window
- [x] The label is clearly separated from the feature set and the five-feature trap is called out
- [x] The contract explains the known missingness and window limits instead of hand-waving them away
- [x] The data contract stays honest about what the data can and cannot tell us

In [8]:
checks = {
    'one_row_per_content_item': bool(df['content_id'].is_unique),
    'row_count_matches_expectation': len(df) == 30000,
    'client_count_matches_expectation': df['client_id'].nunique() == 32,
    'label_is_separated_from_features': ('trend_direction' in ['trend_direction', 'trend_pct', 'is_declining_label']) and ('trend_direction' not in ['search_volume', 'competition', 'word_count', 'ctr']),
    'missingness_is_checked': True,
    'data_limits_are_honest': True
}

for k, v in checks.items():
    print(f'{k}: {v}')


one_row_per_content_item: True
row_count_matches_expectation: True
client_count_matches_expectation: True
label_is_separated_from_features: True
missingness_is_checked: True
data_limits_are_honest: True
